# 🎬 Rick's Clips — High-Speed Google Colab GPU Renderer
Renders all 1,821 frames of your episode at full 1080p (1080x1920) in minutes using Google's free Nvidia T4 Cloud GPU.

---
### ⚡ Quick Start (2 Steps):
1. **Enable Free GPU**: Top menu -> `Runtime` -> `Change runtime type` -> select **T4 GPU** -> click `Save`.
2. **Click Run**: Click the **▶ Play button** on the cell below.

Hardware acceleration (Nvidia GLVND, 4-worker parallel rendering, audio synchronization) is fully automated. When rendering finishes, the MP4 video downloads directly to your PC.

In [ ]:
#@title 🚀 1-Click Render: Select Episode & Click Play
import os, sys, time, json, subprocess, glob
from google.colab import files

EPISODE = "boston-molasses" #@param ["boston-molasses", "lake-peigneur"]
SCALE = 1.0 #@param {type:"slider", min:0.5, max:1.0, step:0.1}
WORKERS = 4 #@param {type:"slider", min:1, max:4, step:1}

print("="*65)
print(f"🎬 STARTING PARALLEL GPU RENDER: {EPISODE.upper()} ({WORKERS} workers @ scale {SCALE})")
print("="*65)

# 1. Verify GPU
print("\n[1/6] 🔍 Checking GPU...")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# 2. Configure Nvidia GLVND for Hardware WebGL Acceleration
print("\n[2/6] ⚙️ Configuring Nvidia EGL GLVND driver for hardware acceleration...")
!echo "/usr/lib64-nvidia" > /etc/ld.so.conf.d/nvidia.conf && ldconfig
!mkdir -p /usr/share/glvnd/egl_vendor.d
with open('/usr/share/glvnd/egl_vendor.d/10_nvidia.json', 'w') as f:
    f.write('''{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "/usr/lib64-nvidia/libEGL_nvidia.so.0"
    }
}''')

os.environ['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia:' + os.environ.get('LD_LIBRARY_PATH', '')
os.environ['__GLX_VENDOR_LIBRARY_NAME'] = 'nvidia'
os.environ['__EGL_VENDOR_LIBRARY_FILENAMES'] = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'

# 3. Install System Dependencies & Playwright
print("\n[3/6] 📦 Installing Node.js, FFmpeg, and Playwright Chromium...")
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
!apt-get install -y nodejs ffmpeg xvfb > /dev/null 2>&1

if not os.path.exists("/content/ricks-clips"):
    !git clone https://github.com/pradeep142536/ricks-clips-00.git /content/ricks-clips
else:
    !git -C /content/ricks-clips pull origin main

!cd /content/ricks-clips/templates/threejs_film && npm install playwright > /dev/null 2>&1
!cd /content/ricks-clips/templates/threejs_film && npx playwright install chromium
!cd /content/ricks-clips/templates/threejs_film && npx playwright install-deps chromium

# 4. Bake Film Plan
print(f"\n[4/6] 📐 Baking film plan for {EPISODE}...")
%cd /content/ricks-clips
!python tools/film_plan.py check episodes/{EPISODE}/film.json

# Read total frame count from baked plan
with open(f'episodes/{EPISODE}/out/plan/info.json', 'r') as f:
    plan_info = json.load(f)
total_frames = plan_info.get('frames', 1821)
fps = plan_info.get('fps', 30)
print(f"Episode plan: {total_frames} frames at {fps} fps")

# 5. Parallel GPU Render across workers
print(f"\n[5/6] 🎥 Launching {WORKERS} parallel GPU render workers for {total_frames} frames...")
t0 = time.time()
slice_size = total_frames // WORKERS
ranges = []
for i in range(WORKERS):
    start = i * slice_size
    end = (i + 1) * slice_size - 1 if i < WORKERS - 1 else total_frames - 1
    ranges.append((start, end))

procs = []
for i, (start, end) in enumerate(ranges):
    chunk_out = f'/content/chunk_{i}.mp4'
    log_out = f'/content/worker_{i}.log'
    cmd = f'xvfb-run -a -s "-screen 0 1080x1920x24" node templates/threejs_film/render.mjs episodes/{EPISODE}/film.json --animatic --range {start}-{end} --scale {SCALE} --no-audio --out {chunk_out} > {log_out} 2>&1'
    p = subprocess.Popen(cmd, shell=True, cwd='/content/ricks-clips')
    procs.append((i, p, chunk_out, log_out, start, end))
    print(f"  Worker {i+1} started: frames {start} -> {end}")

for i, p, chunk_out, log_out, start, end in procs:
    p.wait()
    size_mb = os.path.getsize(chunk_out) / (1024*1024) if os.path.exists(chunk_out) else 0
    print(f"  Worker {i+1} finished (Code {p.returncode}): frames {start}->{end} ({size_mb:.2f} MB)")

render_time = time.time() - t0
print(f"Parallel render completed in {render_time:.1f}s ({render_time/60:.1f}m)!")

# 6. Concatenate & Mux Audio with padding
print(f"\n[6/6] 🎵 Concatenating video chunks and muxing synchronized audio...")
final_video = f"/content/{EPISODE}_full_episode_1080p.mp4"
with open('/content/chunks.txt', 'w') as f:
    for i in range(WORKERS):
        f.write(f"file '/content/chunk_{i}.mp4'\n")

voice_path = f"/content/ricks-clips/episodes/{EPISODE}/voice/voice.wav"
if not os.path.exists(voice_path):
    voice_path = f"/content/ricks-clips/episodes/{EPISODE}/voice/voice.mp3"

if os.path.exists(voice_path):
    concat_cmd = f"ffmpeg -y -f concat -safe 0 -i /content/chunks.txt -i '{voice_path}' -c:v copy -c:a aac -af apad -shortest '{final_video}'"
else:
    concat_cmd = f"ffmpeg -y -f concat -safe 0 -i /content/chunks.txt -c copy '{final_video}'"

subprocess.run(concat_cmd, shell=True, check=True)

# 7. Verification & Download
if os.path.exists(final_video) and os.path.getsize(final_video) > 1000:
    mb = os.path.getsize(final_video) / (1024*1024)
    total_time = time.time() - t0
    print("="*65)
    print(f"🎉 SUCCESS! Full 1080p video generated: {final_video} ({mb:.2f} MB)")
    print(f"⏱️ Total Time: {total_time:.1f}s ({total_time/60:.1f} min)")
    print("⬇️ Downloading video to your computer...")
    print("="*65)
    try:
        files.download(final_video)
    except Exception as e:
        print("Download note:", e)
else:
    print("\n❌ Error: Final video assembly failed.")
